# CPU notebook 4: five seeds, with and without the cold-start-time input (paper, RQ1, RQ6)

Runs `scripts/experiments/run_seed_study.py` at normal and doubled cold-start times.

**Setup.** Attach the code dataset and the data dataset; accelerator None (CPU). *Save Version → Save & Run All*.
Finished seeds are skipped if the notebook is rerun.

In [ ]:
# 1. Find the code and data (any depth under /kaggle/input). No pip install: the code is put on Python's path.
import glob, os, shutil, sys
print('looking for the code ...', flush=True)
hits = [p for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(p), 'src', 'stawarm'))]
print('found:', hits, flush=True)
assert hits, 'Code not found. Check that the stawarm-code dataset is attached (Add Input) and contains pyproject.toml and src/.'
# Use the copy of this repository (it has reproduce.py); older copies in the same dataset are ignored.
current = [p for p in hits if os.path.exists(os.path.join(os.path.dirname(p), 'reproduce.py'))]
assert current, 'The attached stawarm-code dataset holds an older version of the code; upload the current repository as a new version.'
code_dir = os.path.dirname(current[0])
WORK = '/kaggle/working/stawarm'
if not os.path.exists(WORK):
    shutil.copytree(code_dir, WORK)
os.chdir(WORK)
# The repository ships the paper's result files in results/; keep them aside for comparison so that this
# notebook writes only its own outputs to results/.
if os.path.isdir('results') and not os.path.isdir('results_paper'):
    shutil.move('results', 'results_paper')
os.makedirs('results', exist_ok=True)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')          # used by every !python command below
sys.path.insert(0, os.path.join(WORK, 'src'))
os.makedirs('data/processed', exist_ok=True)
for name in ['azure2021_events.parquet', 'huawei_r2_day30_events.parquet', 'huawei_r2_cold_start_stats.parquet']:
    src = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    print(name, '->', src, flush=True)
    assert src, f'{name} not found: check that the stawarm-data dataset is attached'
    if not os.path.exists(f'data/processed/{name}'):
        os.symlink(src[0], f'data/processed/{name}')
import stawarm
print('code from', code_dir, '| package OK:', stawarm.__file__)
print('data:', os.listdir('data/processed'), flush=True)

In [ ]:
# 2. Record the software and hardware environment of this run
!python scripts/tools/record_environment.py results/environment_04_cpu_seed_study.txt

In [ ]:
# 3. Tests (should end with '35 passed')
!python -m pytest -q 2>&1 | tail -2

## 4. Normal cold-start times (5 seeds x 2 settings)

In [ ]:
!python -u scripts/experiments/run_seed_study.py 2>&1 | grep -v Warning | grep -E "seed|Five-seed|plain|coldtime|keep-alive|Paired|cold@|slow@|Learned|wrote" 

## 5. Cold-start times doubled

In [ ]:
!python -u scripts/experiments/run_seed_study.py --scale 2 2>&1 | grep -v Warning | grep -E "seed|Five-seed|plain|coldtime|keep-alive|Paired|cold@|slow@|Learned|wrote" 

In [ ]:
# 6. Compare this run with the paper's result files (zero differences expected for tree-based runs)
!python scripts/analysis/compare_results.py results results_paper --out results/comparison_04_cpu_seed_study.txt

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/results_cpu_five_seeds', 'zip', 'results')
for f in ['results/seedstudy_azure2021/summary.txt', 'results/seedstudy_azure2021_x2/summary.txt']:
    print(open(f).read() if os.path.exists(f) else 'missing ' + f)